# Repli-seq completion bounds on the line and the torus

This notebook is the executable analysis record for the workflow described in the README. Starting from Repli-seq timing tracks, it extracts timing profiles, fits initiation-rate landscapes, runs stochastic replication simulations, and compares completion-time statistics with the analytical bounds developed by Alkhaled, Berkemeier & Nik (2026).

The analysis is split into two ready-to-run workflows.

1. **Non-periodic chromosome profiles.** These are compared with the full-line bound $\mathbb{R}$.
2. **Periodic interval profiles.** These are compared with the torus bound $T_L$.

Shared definitions for Repli-seq extraction, timing-curve polishing, initiation-rate fitting following Berkemeier et al. (2025), stochastic simulation, completion-time bounds, and expected-time bounds are collected first. The two analysis sections can then be run independently, one dataset at a time.


## 1. Shared imports and global settings

Change `DATA_DIR` if your bigWig files are not stored under `data/`.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pybigtools
from IPython.display import display

from repliseq_completion_bounds import plotf, rescale, rfit, rsim

plt.rcParams.update({
    "figure.figsize": (7.2, 4.6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})

warnings.filterwarnings("ignore", category=RuntimeWarning)

DATA_DIR = Path("data")
TIMING_DIR = Path("timing")
FIGURE_DIR = Path("figures")
TIMING_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

S_PHASE_BINS = ["S1", "S2", "S3", "S4", "S5"]
STANDARD_CHROMS = [f"chr{i}" for i in range(1, 23)] + ["chrX"]


## 2. Dataset and file helpers

The periodic-interval analysis uses explicit genomic windows. The line-domain analysis can build one configuration per chromosome from the chromosome sizes stored in the bigWig files.


In [ ]:
def safe_filename(text):
    return (
        str(text)
        .replace(" ", "_")
        .replace("/", "_")
        .replace(",", "")
        .replace(":", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("$", "")
        .replace("\\", "")
        .replace("+", "plus")
    )


def available_cell_lines_from_data(data_dir=DATA_DIR, resolution=10_000,
                                   s_phase_bins=S_PHASE_BINS):
    first_bin = s_phase_bins[0]
    suffix = f"_{first_bin}_{resolution}.bw"
    cell_lines = []

    for path in sorted(data_dir.glob(f"*_{first_bin}_{resolution}.bw")):
        cell_line = path.name[:-len(suffix)]

        if all((data_dir / f"{cell_line}_{sbin}_{resolution}.bw").exists()
               for sbin in s_phase_bins):
            cell_lines.append(cell_line)

    return cell_lines


def get_bigwig_chrom_sizes(cell_line, resolution=10_000, data_dir=DATA_DIR, sbin="S1"):
    path = data_dir / f"{cell_line}_{sbin}_{resolution}.bw"

    if not path.exists():
        raise FileNotFoundError(f"Cannot find {path}")

    bw = pybigtools.open(str(path))

    try:
        chroms = dict(bw.chroms())
    finally:
        bw.close()

    return chroms


def resolve_chrom_name(chrom, chrom_sizes):
    candidates = [chrom]

    if chrom.startswith("chr"):
        candidates.append(chrom[3:])
    else:
        candidates.append(f"chr{chrom}")

    if chrom in {"chrM", "M", "MT"}:
        candidates.extend(["chrM", "M", "MT"])

    for candidate in candidates:
        if candidate in chrom_sizes:
            return candidate

    return None


def require_chrom_name(chrom, chrom_sizes, cell_line, resolution):
    resolved = resolve_chrom_name(chrom, chrom_sizes)

    if resolved is None:
        examples = ", ".join(list(chrom_sizes)[:8])
        raise ValueError(
            f"{chrom} is not present in {cell_line}_S1_{resolution}.bw. "
            f"Available chromosome examples: {examples}"
        )

    return resolved


def build_chromosome_configs(cell_line="DM", chroms=("chr20",), resolution=10_000,
                             data_dir=DATA_DIR, analysis_tag="line"):
    chrom_sizes = get_bigwig_chrom_sizes(
        cell_line=cell_line,
        resolution=resolution,
        data_dir=data_dir,
    )

    configs = {}

    for requested_chrom in chroms:
        chrom = require_chrom_name(
            requested_chrom,
            chrom_sizes=chrom_sizes,
            cell_line=cell_line,
            resolution=resolution,
        )

        chrom_length = int(chrom_sizes[chrom])
        end = chrom_length - (chrom_length % resolution)

        key = f"{cell_line}_{analysis_tag}_{safe_filename(requested_chrom)}"

        configs[key] = {
            "key": key,
            "label": f"{cell_line}, {requested_chrom} line profile",
            "short_label": f"{cell_line} {requested_chrom} line",
            "point_label": f"{cell_line} {requested_chrom}",
            "cell_line": cell_line,
            "chrom": chrom,
            "requested_chrom": requested_chrom,
            "start": 0,
            "end": end,
            "resolution": resolution,
            "fit_periodic": False,
            "sim_periodic": False,
            "bound_geometries": ["line"],
            "analysis_type": "line_profile",
        }

    return configs


def build_chromosome_configs_for_cell_lines(cell_lines, chroms=("chr20",),
                                           resolution=10_000, data_dir=DATA_DIR,
                                           analysis_tag="line"):
    configs = {}

    for cell_line in cell_lines:
        configs.update(build_chromosome_configs(
            cell_line=cell_line,
            chroms=chroms,
            resolution=resolution,
            data_dir=data_dir,
            analysis_tag=analysis_tag,
        ))

    return configs


def build_periodic_interval_configs(cell_lines, regions, resolution=10_000,
                                    data_dir=DATA_DIR):
    if isinstance(cell_lines, str):
        cell_lines = [cell_lines]

    configs = {}

    for cell_line in cell_lines:
        chrom_sizes = get_bigwig_chrom_sizes(
            cell_line=cell_line,
            resolution=resolution,
            data_dir=data_dir,
        )

        for region in regions:
            requested_chrom = region["chrom"]
            chrom = require_chrom_name(
                requested_chrom,
                chrom_sizes=chrom_sizes,
                cell_line=cell_line,
                resolution=resolution,
            )

            region_id = region.get(
                "region_id",
                safe_filename(f"{requested_chrom}_{region['start']}_{region['end']}").upper(),
            )
            label = region.get("label", f"{requested_chrom}:{region['start']}-{region['end']}")
            short_label = region.get("short_label", label)
            key = f"{cell_line}_{region_id}"

            configs[key] = {
                "key": key,
                "label": f"{cell_line}, {label}",
                "short_label": f"{cell_line} {short_label}",
                "point_label": f"{cell_line} {short_label}",
                "cell_line": cell_line,
                "chrom": chrom,
                "requested_chrom": requested_chrom,
                "start": region["start"],
                "end": region["end"],
                "resolution": resolution,
                "fit_periodic": True,
                "sim_periodic": True,
                "bound_geometries": ["torus"],
                "analysis_type": "periodic_interval",
            }

    return configs

## 3. Repli-seq timing extraction

For each genomic bin, the five S-phase bigWig signals are normalised into a probability distribution over S phase. A sigmoid is fitted to the cumulative distribution, and the midpoint is used as a raw timing value. This timing curve is the observed input for the fitted initiation landscape used in both simulations and bound calculations.

In [ ]:
def sigmoid(x, k, x0):
    return 1.0 / (1.0 + np.exp(-k * (x - x0)))


def read_bigwig_bin_mean(bw, chrom, start, end):
    vals = bw.values(chrom, int(start), int(end))

    if vals is None:
        return np.nan

    vals = np.asarray(
        [np.nan if v is None else float(v) for v in vals],
        dtype=float,
    )

    if vals.size == 0:
        return np.nan

    return np.nanmean(vals)


def compute_timing_curve(chrom, start, end, cell_line="DM", resolution=10_000,
                         data_dir=DATA_DIR, s_phase_bins=S_PHASE_BINS):
    x_bins = np.arange(1, len(s_phase_bins) + 1, dtype=float)

    bw_files = [
        pybigtools.open(str(data_dir / f"{cell_line}_{sbin}_{resolution}.bw"))
        for sbin in s_phase_bins
    ]

    positions = np.arange(start, end, resolution, dtype=int)
    rt_values = []

    try:
        for pos in positions:
            signals = np.array([
                read_bigwig_bin_mean(bw, chrom, pos, pos + resolution)
                for bw in bw_files
            ], dtype=float)

            if np.any(~np.isfinite(signals)) or np.nansum(signals) <= 0:
                rt_values.append(np.nan)
                continue

            norm_signals = signals / np.sum(signals)
            cumulative = np.cumsum(norm_signals)

            try:
                popt, _ = curve_fit(
                    sigmoid,
                    x_bins,
                    cumulative,
                    p0=(2.0, 3.0),
                    bounds=([0.1, 1.0], [10.0, len(s_phase_bins)]),
                    maxfev=10_000,
                )
                _, x0 = popt
                rt_values.append(x0)
            except RuntimeError:
                rt_values.append(np.nan)

    finally:
        for bw in bw_files:
            bw.close()

    return positions, np.asarray(rt_values, dtype=float)


def compute_timing_curve_from_config(cfg, data_dir=DATA_DIR):
    return compute_timing_curve(
        chrom=cfg["chrom"],
        start=cfg["start"],
        end=cfg["end"],
        cell_line=cfg["cell_line"],
        resolution=cfg["resolution"],
        data_dir=data_dir,
    )

### Raw timing CSV preprocessing cache

`compute_timing_curve_from_config` returns two arrays: genomic bin starts and a raw timing value per bin. The raw timing value is the fitted sigmoid midpoint in S-phase-bin units, before any smoothing, refinement, or rescaling into minutes. Because extracting this vector from five bigWig files is the slow preprocessing step, the helpers below save one CSV per chromosome or interval under `timing/`. Later analysis cells load those CSVs by default and only rebuild them when the file is missing or `overwrite=True` is requested.

In [ ]:
def timing_cache_stem(cfg):
    chrom = cfg.get("requested_chrom", cfg["chrom"])
    analysis_type = cfg.get("analysis_type", "domain")
    return safe_filename(
        f"{cfg['cell_line']}_{analysis_type}_{chrom}_{cfg['start']}_{cfg['end']}_res{cfg['resolution']}"
    )


def timing_cache_path(cfg, timing_dir=TIMING_DIR):
    return Path(timing_dir) / f"{timing_cache_stem(cfg)}_raw_timing.csv"


def timing_curve_table(cfg, positions, timing_raw):
    positions = np.asarray(positions, dtype=np.int64)
    timing_raw = np.asarray(timing_raw, dtype=float)
    resolution = int(cfg["resolution"])

    return pd.DataFrame({
        "cell_line": cfg["cell_line"],
        "analysis_type": cfg.get("analysis_type", ""),
        "chrom": cfg["chrom"],
        "requested_chrom": cfg.get("requested_chrom", cfg["chrom"]),
        "region_start_bp": int(cfg["start"]),
        "region_end_bp": int(cfg["end"]),
        "resolution_bp": resolution,
        "bin_start_bp": positions,
        "bin_end_bp": positions + resolution,
        "timing_s_phase_midpoint": timing_raw,
    })


def _arrays_from_timing_table(table):
    return (
        table["bin_start_bp"].to_numpy(dtype=int),
        table["timing_s_phase_midpoint"].to_numpy(dtype=float),
    )


def load_timing_curve_csv(path):
    table = pd.read_csv(path)
    required = {"bin_start_bp", "timing_s_phase_midpoint"}
    missing = required.difference(table.columns)

    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")

    return _arrays_from_timing_table(table)


def save_timing_curve_csv(cfg, data_dir=DATA_DIR, timing_dir=TIMING_DIR,
                          overwrite=False):
    path = timing_cache_path(cfg, timing_dir=timing_dir)

    if path.exists() and not overwrite:
        print(f"Timing CSV already exists: {path}")
        return path, pd.read_csv(path)

    positions, timing_raw = compute_timing_curve_from_config(
        cfg,
        data_dir=data_dir,
    )
    table = timing_curve_table(cfg, positions, timing_raw)
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(path, index=False)
    print(f"Wrote raw timing CSV: {path} ({len(table):,} bins)")

    return path, table


def get_timing_curve_from_config(cfg, data_dir=DATA_DIR, timing_dir=TIMING_DIR,
                                 cache_mode="auto", overwrite=False):
    modes = {"auto", "load", "build", "ignore"}

    if cache_mode not in modes:
        raise ValueError(f"cache_mode must be one of {sorted(modes)}")

    if cache_mode == "ignore":
        return compute_timing_curve_from_config(cfg, data_dir=data_dir)

    path = timing_cache_path(cfg, timing_dir=timing_dir)

    if path.exists() and cache_mode in {"auto", "load"} and not overwrite:
        print(f"Loading raw timing CSV: {path}")
        return load_timing_curve_csv(path)

    if cache_mode == "load":
        raise FileNotFoundError(
            f"Raw timing CSV not found: {path}. Run preprocessing first, "
            "or use cache_mode='auto' to build it when missing."
        )

    _, table = save_timing_curve_csv(
        cfg,
        data_dir=data_dir,
        timing_dir=timing_dir,
        overwrite=overwrite or cache_mode == "build",
    )
    return _arrays_from_timing_table(table)


def timing_cache_table(configs, selected_keys=None, timing_dir=TIMING_DIR):
    if selected_keys is None:
        selected_keys = list(configs)

    rows = []

    for key in selected_keys:
        cfg = configs[key]
        path = timing_cache_path(cfg, timing_dir=timing_dir)
        rows.append({
            "dataset_key": key,
            "dataset_label": cfg["label"],
            "raw_timing_csv": str(path),
            "exists": path.exists(),
        })

    return pd.DataFrame(rows)


def preprocess_timing_collection(configs, selected_keys=None, data_dir=DATA_DIR,
                                  timing_dir=TIMING_DIR, overwrite=False):
    if selected_keys is None:
        selected_keys = list(configs)

    rows = []

    for key in selected_keys:
        cfg = configs[key]
        path = timing_cache_path(cfg, timing_dir=timing_dir)

        if path.exists() and not overwrite:
            status = "exists"
        else:
            path, _ = save_timing_curve_csv(
                cfg,
                data_dir=data_dir,
                timing_dir=timing_dir,
                overwrite=overwrite,
            )
            status = "written"

        rows.append({
            "dataset_key": key,
            "dataset_label": cfg["label"],
            "raw_timing_csv": str(path),
            "status": status,
        })

    table = pd.DataFrame(rows)
    display(table)
    return table

## 4. Timing-curve polishing

For periodic interval profiles, the default refinement converts 10 kb bins into a 1 kb grid. For chromosome-scale line-domain simulations, use a smaller refinement factor, usually `1`, otherwise the simulation can become very large.


In [ ]:
def refinef_local(x, resolution_factor=1, mode="smooth"):
    x = np.asarray(x, dtype=float)

    if len(x) == 0:
        return x

    if resolution_factor <= 0:
        raise ValueError("resolution_factor must be positive")

    m = max(1, int(round(len(x) * resolution_factor)))
    u = np.clip(np.arange(m) / resolution_factor, 0, len(x) - 1)

    if mode == "smooth":
        return np.interp(u, np.arange(len(x)), x)

    if mode == "const":
        return x[np.clip(np.floor(u + 0.5).astype(int), 0, len(x) - 1)]

    raise ValueError("mode must be 'smooth' or 'const'")


def smoothf_local(data, window=50):
    data = np.asarray(data, dtype=float)
    n = len(data)

    if n == 0:
        return data

    window = int(max(1, min(window, n)))
    kernel = np.ones(window, dtype=float)

    # Reflect padding avoids strong edge artefacts.
    left = data[1:window + 1][::-1] if n > 1 else data
    right = data[-window - 1:-1][::-1] if n > 1 else data
    padded = np.concatenate([left, data, right])

    smoothed = np.convolve(padded, kernel / kernel.sum(), mode="same")
    return smoothed[len(left):len(left) + n]


def fill_nan_linear(y):
    y = np.asarray(y, dtype=float)
    x = np.arange(len(y))
    ok = np.isfinite(y)

    if ok.sum() < 2:
        raise ValueError("Not enough finite timing values to interpolate.")

    return np.interp(x, x[ok], y[ok])


def prepare_timing_curve(positions_raw, timing_raw, resolution=10_000,
                         refine_factor=10, smooth_window=50,
                         timing_range=(60, 10), slice_stop=None):
    timing_filled = fill_nan_linear(timing_raw)

    timing = refinef_local(timing_filled, resolution_factor=refine_factor)
    timing = smoothf_local(timing, window=smooth_window)
    timing = np.asarray(rescale(timing, timing_range), dtype=float)

    positions = refinef_local(positions_raw, resolution_factor=refine_factor)

    if slice_stop is not None:
        timing = timing[:slice_stop]
        positions = positions[:slice_stop]

    dx_kb = resolution / refine_factor / 1000.0

    return {
        "positions_bp": positions,
        "positions_kb": positions / 1000.0,
        "timing_min": timing,
        "dx_kb": dx_kb,
        "refine_factor": refine_factor,
        "smooth_window": smooth_window,
        "timing_range": timing_range,
    }

## 5. Theoretical completion and expected-time bounds

The analytical comparison follows the completion-bound framework of Alkhaled, Berkemeier & Nik (2026), using the same fitted initiation landscape that drives the stochastic simulations. The bound must use one consistent spatial unit. The simulation receives arrays with no physical coordinates, so the natural model unit is **one array index**. If the Repli-seq input is at 10 kb resolution and we do not refine it, one grid step corresponds to 10 kb. A physical fork speed of 1.4 kb/min must therefore be passed to the simulation as

$$
v = \frac{1.4\ \text{kb/min}}{10\ \text{kb/bin}} = 0.14\ \text{bins/min}.
$$

For periodic interval profiles, `geometry="torus"` is used. For non-periodic chromosome profiles, `geometry="line"` is used with `perQ=False`.

For each tolerance `epsilon`, the notebook compares the theoretical uniform completion-time bound with the corresponding empirical simulation quantile. The same survival estimate is then integrated to obtain an upper bound on the slowest expected local replication time:

$$
\sup_x \mathbb{E}[T(x)] \le \int_0^\infty e^{-F(t)} \, dt.
$$

Below, this integrated quantity is compared with the empirical `max_x E[T(x)]` from simulation.


In [ ]:
GEOMETRY_LABELS = {
    "torus": r"Torus $T_L$",
    "line": r"Full line $\mathbb{R}$",
    "halfline": r"Half-line $\mathbb{R}_+$",
}


def _higher_quantiles(x, q):
    x = np.sort(np.asarray(x, dtype=float), axis=0)
    n = x.shape[0]
    q = np.asarray(q, dtype=float)
    k = np.clip(np.ceil(q * n).astype(int) - 1, 0, n - 1)
    return x[k]


def _width(sigma, L, vmin, geometry="torus"):
    sigma = np.asarray(sigma, dtype=float)

    if geometry == "torus":
        return np.minimum(2.0 * vmin * sigma, L)

    if geometry == "line":
        return 2.0 * vmin * sigma

    if geometry == "halfline":
        return vmin * sigma

    raise ValueError("geometry must be 'torus', 'line', or 'halfline'")

In [ ]:

def _torus_local_mass(I, lengths, dx=1.0):
    """
    Local initiation mass on a torus.

    I is assumed to be an initiation rate per grid site per minute.
    The default dx=1 therefore computes mass by summing fitted bin rates.
    Use dx only if I has first been converted to a density per physical unit.
    """
    I = np.asarray(I, dtype=float)
    lengths = np.atleast_1d(np.asarray(lengths, dtype=float))

    n = len(I)
    L = n * dx
    idx = np.arange(n)

    mass = np.concatenate([I, I])
    prefix = np.concatenate([[0.0], np.cumsum(mass)])

    out = np.empty_like(lengths, dtype=float)

    for k, r in enumerate(lengths):
        if r <= 0:
            out[k] = 0.0
            continue

        if r >= L:
            out[k] = dx * np.sum(I)
            continue

        u = r / dx
        q = int(np.floor(u))
        frac = u - q

        totals = dx * (prefix[idx + q] - prefix[idx])

        if frac > 0:
            totals = totals + frac * dx * I[(idx + q) % n]

        out[k] = np.min(totals)

    return out


def _line_local_mass_finite(I, lengths, dx=1.0):
    """
    Local initiation mass on a finite, non-wrapping profile.

    This is the practical finite-window version used for non-periodic
    chromosome-scale simulations. With the default dx=1, I is
    treated as a fitted rate per grid site per minute, and m_I(r)
    is obtained by summing rates over non-wrapping intervals of grid length r.

    Note that a strict full-line bound would require a model for I(x) outside
    the observed chromosome/window. Here we compare against the observed
    finite profile without circular wrap-around, which matches perQ=False.
    """
    I = np.asarray(I, dtype=float)
    lengths = np.atleast_1d(np.asarray(lengths, dtype=float))

    n = len(I)
    L = n * dx
    prefix = np.concatenate([[0.0], np.cumsum(I)])
    total_mass = dx * np.sum(I)

    out = np.empty_like(lengths, dtype=float)

    for k, r in enumerate(lengths):
        if r <= 0:
            out[k] = 0.0
            continue

        if r >= L:
            out[k] = total_mass
            continue

        u = r / dx
        q = int(np.floor(u))
        frac = u - q

        if frac > 0:
            max_start = n - q - 1
        else:
            max_start = n - q

        if max_start <= 0:
            out[k] = total_mass
            continue

        idx = np.arange(max_start)
        totals = dx * (prefix[idx + q] - prefix[idx])

        if frac > 0:
            totals = totals + frac * dx * I[idx + q]

        out[k] = np.min(totals)

    return out


def _periodic_extension_local_mass(I, lengths, dx=1.0):
    """
    Optional local mass on R using a periodic extension of the fitted landscape.

    This is useful for topology-only comparisons, but is not used by default
    for the non-periodic full-line analysis.
    """
    I = np.asarray(I, dtype=float)
    lengths = np.atleast_1d(np.asarray(lengths, dtype=float))

    n = len(I)
    L = n * dx
    total_mass = dx * np.sum(I)

    out = np.empty_like(lengths, dtype=float)

    for k, r in enumerate(lengths):
        if r <= 0:
            out[k] = 0.0
            continue

        n_periods = int(np.floor(r / L))
        remainder = r - n_periods * L

        out[k] = n_periods * total_mass

        if remainder > 1e-12:
            out[k] += _torus_local_mass(I, [remainder], dx=dx)[0]

    return out


def _local_mass(I, lengths, dx=1.0, geometry="torus", line_extension="finite"):
    if geometry == "torus":
        return _torus_local_mass(I, lengths, dx=dx)

    if geometry in {"line", "halfline"}:
        if line_extension == "finite":
            return _line_local_mass_finite(I, lengths, dx=dx)
        if line_extension == "periodic":
            return _periodic_extension_local_mass(I, lengths, dx=dx)

    raise ValueError("Invalid geometry or line_extension.")


In [ ]:

def completion_survival_exponent_curve(frates, vmin_grid=1.4, dx_grid=1.0, dx_kb=None,
                                       geometry="torus", rhs_max=None, num_t=4000,
                                       line_extension="finite"):
    """
    Build F(t) = int_0^t m_I(w_geometry(sigma)) d sigma.

    Parameters
    ----------
    frates : array
        Fitted initiation rates, interpreted as rates per grid site
        per minute.
    vmin_grid : float
        Fork speed in grid sites per minute.
    dx_grid : float
        Grid spacing in the same units used to define frates. In this notebook
        this should remain 1.0.
    dx_kb : float or None
        Physical size of one grid site in kb. This is stored only to
        label local-mass plots in kb.
    """
    I = np.asarray(frates, dtype=float)

    if I.ndim != 1:
        raise ValueError("frates must be a 1D array")

    if np.any(I < 0):
        raise ValueError("frates must be nonnegative")

    if vmin_grid <= 0 or dx_grid <= 0:
        raise ValueError("vmin_grid and dx_grid must be positive")

    L_grid = len(I) * dx_grid
    total_mass = dx_grid * np.sum(I)

    if total_mass <= 0:
        raise ValueError("Total initiation mass must be positive.")

    if rhs_max is None:
        rhs_max = np.log(1.0 / 1e-4)

    t_wrap = L_grid / (2.0 * vmin_grid)
    t_max = t_wrap + rhs_max / max(total_mass, 1e-12) + dx_grid / vmin_grid

    while True:
        t = np.linspace(0.0, t_max, num_t)
        r_grid = _width(t, L=L_grid, vmin=vmin_grid, geometry=geometry)

        r_unique, inv = np.unique(r_grid, return_inverse=True)
        m_unique = _local_mass(
            I,
            r_unique,
            dx=dx_grid,
            geometry=geometry,
            line_extension=line_extension,
        )
        mI = m_unique[inv]

        dt = np.diff(t)
        F = np.empty_like(t)
        F[0] = 0.0
        F[1:] = np.cumsum(0.5 * (mI[:-1] + mI[1:]) * dt)

        if F[-1] >= rhs_max:
            break

        t_max *= 2.0

    out = {
        "I": I,
        "L_grid": L_grid,
        "dx_grid": dx_grid,
        "dx_kb": dx_kb,
        "vmin_grid": float(vmin_grid),
        "geometry": geometry,
        "geometry_label": GEOMETRY_LABELS[geometry],
        "line_extension": line_extension,
        "t": t,
        "r": r_grid,
        "mI": mI,
        "F": F,
    }

    if dx_kb is not None:
        out["L_kb"] = L_grid * dx_kb / dx_grid
        out["r_kb"] = r_grid * dx_kb / dx_grid
        out["vmin_kb_min"] = vmin_grid * dx_kb / dx_grid

    return out


def completion_time_bound(eps, frates, vmin_grid=1.4, dx_grid=1.0,
                          dx_kb=None, geometry="torus", num_t=4000,
                          line_extension="finite"):
    eps = np.asarray(eps, dtype=float)

    if np.any((eps <= 0) | (eps >= 1)):
        raise ValueError("eps must lie strictly in (0, 1).")

    rhs = np.log(1.0 / eps)

    aux = completion_survival_exponent_curve(
        frates=frates,
        vmin_grid=vmin_grid,
        dx_grid=dx_grid,
        dx_kb=dx_kb,
        geometry=geometry,
        rhs_max=float(np.max(rhs)),
        num_t=num_t,
        line_extension=line_extension,
    )

    T_bound = np.interp(rhs, aux["F"], aux["t"])

    return T_bound, aux


In [ ]:
def empirical_completion_curve(rep_times_per_sim, eps_grid):
    """
    Empirical curve to compare with the uniform survival bound:
    max_x quantile_{1-eps} T(x), matching the L-infinity survival logic.
    """
    tau = np.asarray(rep_times_per_sim, dtype=float)

    if tau.ndim != 2:
        raise ValueError("rep_times_per_sim must have shape (n_sims, n_pos).")

    q = 1.0 - np.asarray(eps_grid, dtype=float)

    return _higher_quantiles(tau, q).max(axis=1)


def empirical_expected_time(rep_times_per_sim):
    """
    Empirical max_x E[T(x)], matching the integrated uniform survival bound.
    """
    tau = np.asarray(rep_times_per_sim, dtype=float)

    if tau.ndim != 2:
        raise ValueError("rep_times_per_sim must have shape (n_sims, n_pos).")

    return float(np.nanmax(np.nanmean(tau, axis=0)))


def expected_time_bound_from_curve(aux):
    """
    Integrate the survival upper bound to obtain an expected-time bound.

    The tail beyond the last tabulated time is bounded using the final slope
    of F, which is valid because the local-mass curve is non-decreasing.
    """
    t = np.asarray(aux["t"], dtype=float)
    F = np.asarray(aux["F"], dtype=float)
    mI = np.asarray(aux["mI"], dtype=float)

    survival_upper = np.exp(-F)
    trapz = getattr(np, "trapezoid", np.trapz)
    body = float(trapz(survival_upper, t))
    tail_rate = max(float(mI[-1]), 1e-12)
    tail = float(survival_upper[-1] / tail_rate)
    total = body + tail

    return {
        "survival_upper": survival_upper,
        "expected_time_bound": total,
        "expected_time_bound_body": body,
        "expected_time_bound_tail": tail,
        "expected_time_bound_tail_fraction": tail / total if total > 0 else 0.0,
        "expected_time_tail_rate": tail_rate,
    }


def compare_completion_bounds(frates, rep_times_per_sim=None, fork_speed_grid=1.4,
                              dx_grid=1.0, dx_kb=None, eps_grid=None,
                              geometry="torus", num_t=4000,
                              line_extension="finite"):
    if eps_grid is None:
        eps_grid = np.geomspace(1e-4, 0.99, 100)

    eps_grid = np.asarray(eps_grid, dtype=float)

    T_theory, aux = completion_time_bound(
        eps=eps_grid,
        frates=frates,
        vmin_grid=fork_speed_grid,
        dx_grid=dx_grid,
        dx_kb=dx_kb,
        geometry=geometry,
        num_t=num_t,
        line_extension=line_extension,
    )

    out = {
        "eps": eps_grid,
        "T_theory": T_theory,
        "aux": aux,
        "geometry": geometry,
        "geometry_label": GEOMETRY_LABELS[geometry],
        "line_extension": line_extension,
        "fork_speed_grid": fork_speed_grid,
        "dx_grid": dx_grid,
        "dx_kb": dx_kb,
    }

    out.update(expected_time_bound_from_curve(aux))

    if dx_kb is not None:
        out["fork_speed_kb_min"] = fork_speed_grid * dx_kb / dx_grid

    if rep_times_per_sim is not None:
        out["T_empirical"] = empirical_completion_curve(
            rep_times_per_sim,
            eps_grid=eps_grid,
        )
        out["expected_time_empirical_pointwise"] = empirical_expected_time(
            rep_times_per_sim,
        )

    return out


## 6. Plotting helpers

These functions keep the timing and initiation-rate plots, then add the theoretical completion-bound comparisons, expected-time summaries, and tightness diagnostics.


In [ ]:
def plot_replicated_fraction_map(rep_times_per_sim, positions_kb=None,
                                 nt=300, ylims=None, title=None):
    tau = np.asarray(rep_times_per_sim, dtype=float)

    if tau.ndim != 2:
        raise ValueError("rep_times_per_sim must have shape (n_sims, n_pos).")

    n_sims, n_pos = tau.shape

    if positions_kb is None:
        x = np.arange(n_pos)
        xlabel = "Position"
    else:
        x = np.asarray(positions_kb, dtype=float)
        xlabel = "Chromosome position (kb)"

    tmin = max(0.0, np.nanmin(tau))
    tmax = np.nanmax(tau)
    t_grid = np.linspace(tmin, tmax, nt)

    S = (tau[None, :, :] <= t_grid[:, None, None]).mean(axis=1)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    im = ax.imshow(
        S,
        aspect="auto",
        origin="lower",
        extent=[x[0], x[-1], t_grid[0], t_grid[-1]],
        vmin=0.0,
        vmax=1.0,
    )

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(r"Replicated fraction $f(x,t)$")

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Replication time (min)")
    ax.set_title(title or r"Empirical replicated fraction $f(x,t)$")

    if ylims is not None:
        ax.set_ylim(*ylims)
    else:
        ax.invert_yaxis()

    plt.tight_layout()
    return fig, ax, x, t_grid, S


def plot_geometry_bounds(results_by_geometry, title=None, empiricalQ=True):
    fig, ax = plt.subplots(figsize=(6.8, 4.6))

    for geometry, res in results_by_geometry.items():
        ax.plot(
            res["eps"],
            res["T_theory"],
            label=f"{GEOMETRY_LABELS[geometry]} bound",
        )

    if empiricalQ:
        first_res = next(iter(results_by_geometry.values()))

        if "T_empirical" in first_res:
            ax.plot(
                first_res["eps"],
                first_res["T_empirical"],
                linestyle="--",
                label="empirical simulation",
            )

    ax.set_xscale("log")
    ax.set_xlabel(r"Tolerance $\varepsilon$")
    ax.set_ylabel(r"Completion time $T_\varepsilon$ (min)")
    ax.set_title(title or "Completion-time bound")
    ax.legend()
    plt.tight_layout()

    return fig, ax


def plot_expected_time_bounds(results_by_geometry, title=None):
    items = list(results_by_geometry.items())
    labels = {
        "torus": "Torus",
        "line": "R",
        "halfline": "R+",
    }
    theory = np.array([res["expected_time_bound"] for _, res in items], dtype=float)
    empirical = np.array([
        res.get("expected_time_empirical_pointwise", np.nan)
        for _, res in items
    ], dtype=float)
    valid = np.isfinite(empirical) & np.isfinite(theory)

    if not np.any(valid):
        raise ValueError("Expected-time plot requires empirical pointwise simulation estimates.")

    fig, ax = plt.subplots(figsize=(5.4, 5.0))
    ax.scatter(empirical[valid], theory[valid], color="black", zorder=3)

    for (geometry, res), x_val, y_val, keep in zip(items, empirical, theory, valid):
        if keep:
            ax.annotate(
                labels.get(geometry, res["geometry_label"]),
                xy=(x_val, y_val),
                xytext=(6, 6),
                textcoords="offset points",
            )

    lo = float(min(np.min(empirical[valid]), np.min(theory[valid])))
    hi = float(max(np.max(empirical[valid]), np.max(theory[valid])))
    pad = 0.05 * (hi - lo) if hi > lo else 1.0
    lo = max(0.0, lo - pad)
    hi = hi + pad
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1, color="0.5")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

    ax.set_xlabel(r"Empirical $\max_x \mathbb{E}[T(x)]$ (min)")
    ax.set_ylabel("Theoretical expected-time bound (min)")
    ax.set_title(title or "Expected local replication-time bound")
    ax.set_aspect("equal", adjustable="box")
    plt.tight_layout()

    return fig, ax


def expected_time_pair_table(results):
    labels = {
        "torus": "Torus",
        "line": "R",
        "halfline": "R+",
    }
    rows = []

    for key, result in results.items():
        cfg = result["config"]

        for geometry, bound_result in result["bounds"].items():
            rows.append({
                "dataset_key": key,
                "cell_line": cfg["cell_line"],
                "point_label": cfg.get("point_label", cfg.get("short_label", cfg["label"])),
                "dataset_label": cfg["label"],
                "geometry": geometry,
                "geometry_label": labels.get(geometry, bound_result.get("geometry_label", geometry)),
                "E_empirical_pointwise_min": bound_result.get("expected_time_empirical_pointwise", np.nan),
                "E_theory_min": bound_result["expected_time_bound"],
            })

    return pd.DataFrame(rows)


def _resolve_axis_limits(requested, fallback):
    if requested is None:
        return fallback

    if len(requested) != 2:
        raise ValueError("Axis limits must be a (min, max) pair.")

    lo = fallback[0] if requested[0] is None else requested[0]
    hi = fallback[1] if requested[1] is None else requested[1]

    return (lo, hi)


def plot_expected_time_pair_scatter(results, title=None, label_col="point_label",
                                    xlims=None, ylims=None,
                                    equal_aspect=True):
    table = expected_time_pair_table(results)
    xcol = "E_empirical_pointwise_min"
    ycol = "E_theory_min"
    valid = np.isfinite(table[xcol]) & np.isfinite(table[ycol])
    plot_table = table.loc[valid].copy()

    if plot_table.empty:
        raise ValueError("Expected-time scatter requires empirical pointwise simulation estimates.")

    if label_col not in plot_table.columns:
        raise ValueError(f"label_col={label_col!r} is not a column in the expected-time table.")

    fig, ax = plt.subplots(figsize=(5.4, 5.0))

    plot_table["_scatter_label"] = plot_table[label_col].astype(str)

    for scatter_label, group in plot_table.groupby("_scatter_label", sort=False):
        ax.scatter(group[xcol], group[ycol], label=scatter_label, zorder=3)

    lo = float(min(plot_table[xcol].min(), plot_table[ycol].min()))
    hi = float(max(plot_table[xcol].max(), plot_table[ycol].max()))
    pad = 0.05 * (hi - lo) if hi > lo else 1.0
    lo = max(0.0, lo - pad)
    hi = hi + pad

    auto_limits = (lo, hi)
    xlims = _resolve_axis_limits(xlims, auto_limits)
    ylims = _resolve_axis_limits(ylims, auto_limits)

    ref_lo = min(xlims[0], ylims[0])
    ref_hi = max(xlims[1], ylims[1])
    ax.plot([ref_lo, ref_hi], [ref_lo, ref_hi], linestyle="--", linewidth=1, color="0.5")
    ax.set_xlim(xlims)
    ax.set_ylim(ylims)
    ax.set_xlabel(r"Empirical $\max_x \mathbb{E}[T(x)]$ (min)")
    ax.set_ylabel("Theoretical expected-time bound (min)")
    ax.set_title(title or "Expected local replication-time bound")
    if equal_aspect:
        ax.set_aspect("equal", adjustable="box")
    ax.legend(title="Dataset")
    plt.tight_layout()

    return fig, ax, table


def plot_bound_tightness(result, title=None):
    if "T_empirical" not in result:
        raise ValueError("This result has no empirical simulation curve.")

    eps = result["eps"]
    theory = result["T_theory"]
    empirical = result["T_empirical"]

    fig, ax = plt.subplots(figsize=(6.8, 4.2))
    ax.plot(eps, theory / empirical)
    ax.axhline(1.0, linestyle="--", linewidth=1)

    ax.set_xscale("log")
    ax.set_xlabel(r"Tolerance $\varepsilon$")
    ax.set_ylabel(r"Theoretical / empirical $T_\varepsilon$")
    ax.set_title(title or f"Bound tightness: {result['geometry_label']}")
    plt.tight_layout()

    return fig, ax


def plot_local_mass(result, title=None):
    aux = result["aux"]

    fig, ax = plt.subplots(figsize=(6.8, 4.2))
    x = aux["r_kb"] if "r_kb" in aux else aux["r"]
    xlabel = "Arc or interval length r (kb)" if "r_kb" in aux else "Arc or interval length r (grid units)"
    ax.plot(x, aux["mI"])

    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"Local initiation mass $m_I(r)$")
    ax.set_title(title or f"Local initiation-mass function: {aux['geometry_label']}")
    plt.tight_layout()

    return fig, ax


## 7. Core analysis functions

`run_single_dataset` does the numerical work but does not plot. `make_standard_plots` then produces and saves the plots. This makes it easier to run the line-domain and torus-domain analyses independently.


In [ ]:

def run_single_dataset(cfg, fork_speed_kb_min=1.4, sim_number=1000,
                       refine_factor=10, smooth_window=50,
                       timing_range=(60, 10), eps_grid=None,
                       slice_stop=None, num_t_bound=4000,
                       max_rep_time=2000.0, timing_cache_mode="auto",
                       timing_dir=TIMING_DIR,
                       overwrite_timing_cache=False):
    if eps_grid is None:
        eps_grid = np.geomspace(1e-4, 0.99, 100)

    print(f"Processing: {cfg['label']}")
    print(f"Region: {cfg['chrom']}:{cfg['start']}-{cfg['end']}")
    print(f"Model periodicity: fit={cfg['fit_periodic']}, simulation={cfg['sim_periodic']}")
    print(f"Bound geometries: {cfg['bound_geometries']}")

    positions_raw, timing_raw = get_timing_curve_from_config(
        cfg,
        timing_dir=timing_dir,
        cache_mode=timing_cache_mode,
        overwrite=overwrite_timing_cache,
    )

    prepared = prepare_timing_curve(
        positions_raw=positions_raw,
        timing_raw=timing_raw,
        resolution=cfg["resolution"],
        refine_factor=refine_factor,
        smooth_window=smooth_window,
        timing_range=timing_range,
        slice_stop=slice_stop,
    )

    timedata = prepared["timing_min"]
    dx_kb = prepared["dx_kb"]

    # The simulation works in array indices, not physical kb.
    # Convert the physical speed in kb/min into grid sites/min.
    fork_speed_grid = fork_speed_kb_min / dx_kb

    print(f"Grid spacing: {dx_kb:g} kb per simulation site")
    print(f"Fork speed: {fork_speed_kb_min:g} kb/min = {fork_speed_grid:g} grid sites/min")

    frates = rfit(
        "replication_timing",
        "firing_rate",
        source=timedata,
        fork_speed=fork_speed_grid,
        maxiter=10,
        fit_step=2,
        perQ=cfg["fit_periodic"],
    )

    simres = rsim(
        ori_rate=frates,
        fork_speed=fork_speed_grid,
        sim_number=sim_number,
        perQ=cfg["sim_periodic"],
        time_statsQ=True,
        time_stats_xtQ=False,
        time_stats_densQ=None,
        max_rep_time=max_rep_time,
    )

    rep_times_per_sim = simres["rep_times_per_sim"]

    bounds = {}

    for geometry in cfg["bound_geometries"]:
        bounds[geometry] = compare_completion_bounds(
            frates=frates,
            rep_times_per_sim=rep_times_per_sim,
            fork_speed_grid=fork_speed_grid,
            dx_grid=1.0,
            dx_kb=dx_kb,
            eps_grid=eps_grid,
            geometry=geometry,
            num_t=num_t_bound,
            line_extension=cfg.get("line_extension", "finite"),
        )

    return {
        "config": cfg,
        "positions_raw": positions_raw,
        "timing_raw": timing_raw,
        "prepared": prepared,
        "timedata": timedata,
        "frates": frates,
        "simres": simres,
        "rep_times_per_sim": rep_times_per_sim,
        "bounds": bounds,
        "fork_speed_kb_min": fork_speed_kb_min,
        "fork_speed_grid": fork_speed_grid,
        "sim_number": sim_number,
        "max_rep_time": max_rep_time,
        "timing_cache_mode": timing_cache_mode,
        "timing_cache_path": str(timing_cache_path(cfg, timing_dir=timing_dir)),
        "eps_grid": eps_grid,
    }


In [ ]:
def _positive_percentile_limits(values, percentiles=(1, 99.5), log_pad_fraction=0.12):
    if percentiles is None:
        return (None, None)

    finite_positive = np.asarray(values, dtype=float)
    finite_positive = finite_positive[np.isfinite(finite_positive) & (finite_positive > 0)]

    if finite_positive.size == 0:
        return (None, None)

    lo, hi = np.percentile(finite_positive, percentiles)

    if not np.isfinite(lo) or not np.isfinite(hi) or lo <= 0 or hi <= 0:
        return (None, None)

    if lo == hi:
        return (lo / 10.0, hi * 10.0)

    log_lo, log_hi = np.log10([lo, hi])
    pad = log_pad_fraction * max(log_hi - log_lo, 1e-12)

    return (10.0 ** (log_lo - pad), 10.0 ** (log_hi + pad))


def make_standard_plots(result, save_figures=True,
                        initiation_xlims=(None, None),
                        initiation_ylims=None,
                        initiation_ylim_percentiles=(1, 99.5),
                        initiation_invert_y=False):
    cfg = result["config"]
    prefix = safe_filename(cfg["short_label"])

    positions_kb = result["prepared"]["positions_kb"]
    timedata = result["timedata"]
    frates = result["frates"]
    simres = result["simres"]
    rep_times_per_sim = result["rep_times_per_sim"]
    bounds = result["bounds"]

    # Initiation-rate plot.
    if initiation_ylims is None:
        initiation_ylims = _positive_percentile_limits(
            frates,
            percentiles=initiation_ylim_percentiles,
        )

    plotf(
        frates,
        logyQ=True,
        x_array=positions_kb,
        invyQ=initiation_invert_y,
        xlims=initiation_xlims,
        ylims=initiation_ylims,
        xtitle="Chromosome position (kb)",
        ytitle="Initiation rate",
        labels=[cfg["label"]],
        saveQ=False,
    )
    if save_figures:
        plt.savefig(FIGURE_DIR / f"{prefix}_initiation_rate.pdf", bbox_inches="tight")
    plt.show()

    # Repli-seq versus simulation timing plot.
    plotf(
        timedata,
        simres["replication_timing"],
        x_array=positions_kb,
        invyQ=False,
        xtitle="Chromosome position (kb)",
        ytitle="Replication timing (min)",
        labels=["Repli-seq", "Simulation"],
        saveQ=False,
    )
    if save_figures:
        plt.savefig(FIGURE_DIR / f"{prefix}_timing_repliseq_vs_simulation.pdf", bbox_inches="tight")
    plt.show()
    '''
    # Empirical replicated fraction map.
    fig, ax, *_ = plot_replicated_fraction_map(
        rep_times_per_sim,
        positions_kb=positions_kb,
        nt=300,
        title=f"{cfg['label']}: empirical replicated fraction",
    )
    if save_figures:
        fig.savefig(FIGURE_DIR / f"{prefix}_replicated_fraction_map.pdf", bbox_inches="tight")
    plt.show()
    '''
    # Bound versus empirical simulation.
    fig, ax = plot_geometry_bounds(
        bounds,
        title=f"{cfg['label']}: theoretical bound versus simulation",
        empiricalQ=True,
    )
    if save_figures:
        fig.savefig(FIGURE_DIR / f"{prefix}_bound_vs_simulation.pdf", bbox_inches="tight")
    plt.show()

    # Expected-time summary derived from the same survival bound.
    fig, ax = plot_expected_time_bounds(
        bounds,
        title=f"{cfg['label']}: expected local replication-time bound",
    )
    if save_figures:
        fig.savefig(FIGURE_DIR / f"{prefix}_expected_time_bound.pdf", bbox_inches="tight")
    plt.show()

    # Tightness and local-mass plots for each geometry.
    for geometry, bound_result in bounds.items():
        fig, ax = plot_bound_tightness(
            bound_result,
            title=f"{cfg['label']}: tightness of {GEOMETRY_LABELS[geometry]} bound",
        )
        if save_figures:
            fig.savefig(FIGURE_DIR / f"{prefix}_bound_tightness_{geometry}.pdf", bbox_inches="tight")
        plt.show()

        fig, ax = plot_local_mass(
            bound_result,
            title=f"{cfg['label']}: $m_I(r)$ for {GEOMETRY_LABELS[geometry]}",
        )
        if save_figures:
            fig.savefig(FIGURE_DIR / f"{prefix}_local_mass_{geometry}.pdf", bbox_inches="tight")
        plt.show()


def run_dataset_collection(configs, selected_keys, **kwargs):
    """
    Optional batch helper. Use this only when you are ready to loop over several datasets.
    """
    results = {}

    for key in selected_keys:
        results[key] = run_single_dataset(configs[key], **kwargs)

    return results

In [ ]:
def completion_summary_table(results, eps_values=(0.1, 0.05, 0.01, 0.001)):
    rows = []

    for key, result in results.items():
        cfg = result["config"]

        for geometry, bound_result in result["bounds"].items():
            eps_grid = bound_result["eps"]

            for eps in eps_values:
                row = {
                    "dataset_key": key,
                    "dataset_label": cfg["label"],
                    "geometry": geometry,
                    "eps": eps,
                    "dx_kb": result.get("prepared", {}).get("dx_kb", np.nan),
                    "fork_speed_kb_min": result.get("fork_speed_kb_min", np.nan),
                    "fork_speed_grid_per_min": result.get("fork_speed_grid", np.nan),
                    "T_theory_min": np.interp(eps, eps_grid, bound_result["T_theory"]),
                }

                if "T_empirical" in bound_result:
                    row["T_empirical_min"] = np.interp(
                        eps,
                        eps_grid,
                        bound_result["T_empirical"],
                    )
                    row["theory_over_empirical"] = row["T_theory_min"] / row["T_empirical_min"]

                rows.append(row)

    return pd.DataFrame(rows)


def expected_time_summary_table(results):
    rows = []

    for key, result in results.items():
        cfg = result["config"]

        for geometry, bound_result in result["bounds"].items():
            row = {
                "dataset_key": key,
                "dataset_label": cfg["label"],
                "geometry": geometry,
                "dx_kb": result.get("prepared", {}).get("dx_kb", np.nan),
                "fork_speed_kb_min": result.get("fork_speed_kb_min", np.nan),
                "fork_speed_grid_per_min": result.get("fork_speed_grid", np.nan),
                "E_theory_min": bound_result["expected_time_bound"],
                "tail_fraction": bound_result["expected_time_bound_tail_fraction"],
            }

            if "expected_time_empirical_pointwise" in bound_result:
                row["E_empirical_pointwise_min"] = bound_result["expected_time_empirical_pointwise"]
                row["theory_over_empirical_pointwise"] = (
                    row["E_theory_min"] / row["E_empirical_pointwise_min"]
                )

            rows.append(row)

    return pd.DataFrame(rows)


# Analysis A: full-line chromosome profiles

This section analyses non-periodic chromosome-scale timing profiles. The fit and simulation use `perQ=False`, and the theoretical comparison uses the full-line completion bound. Because chromosome-wide simulations can be large, the default run is organised one profile at a time.


## A1. Preprocess or load full-line raw timing

Choose the cell lines and chromosomes to include in the full-line analysis, then build or load one raw timing CSV per selected pair. This is the only step that needs the bigWig files. Later cells read the cached CSVs from `timing/`.

In [ ]:
LINE_CELL_LINES = ["DM", "HSR", "PC3DM", "RPE1"]
LINE_CHROMS = ["chr1"]
LINE_PROFILE_DATASETS = build_chromosome_configs_for_cell_lines(
    cell_lines=LINE_CELL_LINES,
    chroms=LINE_CHROMS,
    resolution=10_000,
    data_dir=DATA_DIR,
    analysis_tag="line",
)
display(pd.DataFrame([
    {
        "key": key,
        "cell_line": cfg["cell_line"],
        "chromosome": cfg.get("requested_chrom", cfg["chrom"]),
        "resolved_chromosome": cfg["chrom"],
        "region": f"{cfg['chrom']}:{cfg['start']}-{cfg['end']}",
    }
    for key, cfg in LINE_PROFILE_DATASETS.items()
]))

display(timing_cache_table(LINE_PROFILE_DATASETS))

line_timing_preprocessing = preprocess_timing_collection(
    LINE_PROFILE_DATASETS,
    overwrite=False,
)

## A2. Run one full-line chromosome analysis

Choose one preprocessed full-line dataset and run it from the cached raw timing CSV. For chromosome-scale simulations, `refine_factor=1` keeps the grid at 10 kb and avoids very large simulations. The simulation is non-periodic: `perQ=False`.

The speed is entered below in physical units, `fork_speed_kb_min=1.4`. `run_single_dataset` converts it internally using the actual grid spacing and prints the converted value. With a 10 kb grid this becomes `0.14` grid sites/min. Do not manually divide the speed before passing it here.

In [ ]:
LINE_EXAMPLE_CELL_LINE = "DM"
LINE_EXAMPLE_CHROM = "chr1"
LINE_KEY = f"{LINE_EXAMPLE_CELL_LINE}_line_{safe_filename(LINE_EXAMPLE_CHROM)}"

if LINE_KEY not in LINE_PROFILE_DATASETS:
    available = ", ".join(LINE_PROFILE_DATASETS)
    raise KeyError(f"{LINE_KEY} is not available. Available keys: {available}")

line_result = run_single_dataset(
    LINE_PROFILE_DATASETS[LINE_KEY],
    fork_speed_kb_min=1.4,
    sim_number=1_000,          # reduce during exploratory runs if needed
    refine_factor=1,           # keep the line-profile grid manageable
    smooth_window=25,
    timing_range=(450, 30),
    eps_grid=np.geomspace(1e-4, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

## A3. Plot and summarize the selected full-line result

Plot the standard diagnostics for the single full-line result from A2.

In [ ]:
make_standard_plots(
    line_result,
    save_figures=True,
    initiation_ylims=(1e-4, 1e-2),
    # initiation_ylim_percentiles=None,  # use manual y-limits exactly
)

expected_time_summary_table({LINE_KEY: line_result})

## A4. Full-line batch scatter across selected pairs

Run every full-line pair selected in A1 and show the empirical-versus-theoretical expected-time scatter.

In [ ]:
LINE_BATCH_DATASETS = LINE_PROFILE_DATASETS

line_batch_results = run_dataset_collection(
    LINE_BATCH_DATASETS,
    selected_keys=list(LINE_BATCH_DATASETS),
    fork_speed_kb_min=1.4,
    sim_number=10_000,
    refine_factor=1,
    smooth_window=25,
    timing_range=(450, 30),
    eps_grid=np.geomspace(1e-4, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

In [ ]:
fig, ax, line_expected_time_pairs = plot_expected_time_pair_scatter(
    line_batch_results,
    title="Full line: expected local replication-time bound across selected pairs",
    xlims=(0,650),
    ylims=(0,650),
)
fig.savefig(FIGURE_DIR / "line_expected_time_pairs.pdf", bbox_inches="tight")
plt.show()

# Analysis B: periodic intervals on the torus

This section treats selected genomic windows as periodic domains. The fit and simulation use `perQ=True`, and the theoretical comparison uses the torus completion bound. The coordinates can be changed freely; the point is to compare a periodic window with the torus estimate without tying the example to a specific biological interpretation.


## B1. Preprocess or load periodic-interval raw timing

Choose the cell lines and periodic genomic regions to include, then build or load one raw timing CSV per selected pair. The periodic fit and simulation are applied later; this cache only stores the extracted one-dimensional timing vector from the bigWig files.

In [ ]:
PERIODIC_CELL_LINES = ["DM", "HSR", "PC3DM", "RPE1"]
PERIODIC_REGIONS = [
    {
        "region_id": "CHR8_PERIODIC_INTERVAL",
        "label": "chr8 periodic interval",
        "short_label": "chr8 interval",
        "chrom": "chr8",
        "start": 126_425_747,
        "end": 127_997_820,
    },
    {
        "region_id": "CHR8_UPSTREAM_WINDOW",
        "label": "chr8 upstream periodic window",
        "short_label": "chr8 upstream",
        "chrom": "chr8",
        "start": 124_000_000,
        "end": 125_600_000,
    },
    {
        "region_id": "CHR8_DOWNSTREAM_WINDOW",
        "label": "chr8 downstream periodic window",
        "short_label": "chr8 downstream",
        "chrom": "chr8",
        "start": 128_800_000,
        "end": 130_400_000,
    },
    {
        "region_id": "CHR8_DISTAL_WINDOW",
        "label": "chr8 distal periodic window",
        "short_label": "chr8 distal",
        "chrom": "chr8",
        "start": 131_000_000,
        "end": 132_600_000,
    },
]

PERIODIC_INTERVAL_DATASETS = build_periodic_interval_configs(
    cell_lines=PERIODIC_CELL_LINES,
    regions=PERIODIC_REGIONS,
    resolution=10_000,
    data_dir=DATA_DIR,
)
display(pd.DataFrame([
    {
        "key": key,
        "cell_line": cfg["cell_line"],
        "region": cfg["short_label"].replace(f"{cfg['cell_line']} ", ""),
        "coordinates": f"{cfg.get('requested_chrom', cfg['chrom'])}:{cfg['start']}-{cfg['end']}",
        "resolved_chromosome": cfg["chrom"],
    }
    for key, cfg in PERIODIC_INTERVAL_DATASETS.items()
]))

display(timing_cache_table(PERIODIC_INTERVAL_DATASETS))

periodic_timing_preprocessing = preprocess_timing_collection(
    PERIODIC_INTERVAL_DATASETS,
    overwrite=False,
)

## B2. Run one periodic-interval analysis

Choose one preprocessed periodic interval and run it from the cached raw timing CSV. This uses a refined 1 kb grid by default, periodic simulations, and the same physical-speed input convention as the full-line analysis.

In [ ]:
PERIODIC_EXAMPLE_CELL_LINE = "DM"
PERIODIC_EXAMPLE_REGION_ID = "CHR8_PERIODIC_INTERVAL"
PERIODIC_KEY = f"{PERIODIC_EXAMPLE_CELL_LINE}_{PERIODIC_EXAMPLE_REGION_ID}"

if PERIODIC_KEY not in PERIODIC_INTERVAL_DATASETS:
    available = ", ".join(PERIODIC_INTERVAL_DATASETS)
    raise KeyError(f"{PERIODIC_KEY} is not available. Available keys: {available}")

periodic_result = run_single_dataset(
    PERIODIC_INTERVAL_DATASETS[PERIODIC_KEY],
    fork_speed_kb_min=1.4,
    sim_number=10_000,          # reduce during exploratory runs if needed
    refine_factor=10,          # 10 kb input -> 1 kb simulation grid
    smooth_window=50,
    timing_range=(60, 10),
    eps_grid=np.geomspace(1e-4, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

## B3. Plot and summarize the selected periodic result

Plot the standard diagnostics for the single periodic result from B2.

In [ ]:
make_standard_plots(
    periodic_result,
    save_figures=True,
    # initiation_ylims=(1e-8, 1e-3),
    # initiation_ylim_percentiles=None,  # use manual y-limits exactly
)

expected_time_summary_table({PERIODIC_KEY: periodic_result})

## B4. Periodic batch scatter across selected pairs

Run every periodic pair selected in B1 and show the empirical-versus-theoretical expected-time scatter.

In [ ]:
PERIODIC_BATCH_DATASETS = PERIODIC_INTERVAL_DATASETS

periodic_batch_results = run_dataset_collection(
    PERIODIC_BATCH_DATASETS,
    selected_keys=list(PERIODIC_BATCH_DATASETS),
    fork_speed_kb_min=1.4,
    sim_number=1000,
    refine_factor=10,
    smooth_window=50,
    timing_range=(60, 10),
    eps_grid=np.geomspace(1e-4, 0.99, 100),
    num_t_bound=4_000,
    max_rep_time=2_000,
    timing_cache_mode="load",
)

In [ ]:
fig, ax, periodic_expected_time_pairs = plot_expected_time_pair_scatter(
    periodic_batch_results,
    title="Torus: expected local replication-time bound across selected pairs",
    xlims=(0,100),
    ylims=(0,100),
)
fig.savefig(FIGURE_DIR / "torus_expected_time_pairs.pdf", bbox_inches="tight")
plt.show()

# Notes on interpretation

For non-periodic chromosome profiles, the simulation is run with `perQ=False` and the comparison is made with the full-line completion bound. The local initiation mass is computed using non-wrapping intervals in the observed chromosome, not circular arcs. This keeps the analysis aligned with the line geometry.

For periodic interval profiles, the torus completion bound is the natural theoretical object because the window is being analysed as a circular domain. The fit and stochastic simulations are therefore run with `perQ=True`.

In both sections, the code converts the physical fork speed in kb/min into grid units using

$$
v_{\text{grid}} = \frac{v_{\text{kb/min}}}{dx_{\text{kb}}}.
$$

and uses this grid speed in both the stochastic simulations and the theoretical bound. The local initiation mass is computed in grid units, so fitted initiation rates are not multiplied by the 10 kb bin size. The kb conversion is used only for plotting lengths.

The expected-time summary is obtained by integrating the same survival upper bound used for the completion-time curves. It therefore matches the pointwise-uniform logic and bounds the slowest expected local replication time `max_x E[T(x)]`. For full citation details, see the README references: initiation-rate fitting follows Berkemeier et al. (2025), and the completion-bound comparison follows Alkhaled, Berkemeier & Nik (2026).
